# Grid Mapper – RF-DETR + SatlasPretrain continual-learning training

This notebook combines Grid Mapper's African training/review data with an **optional SatlasPretrain bootstrap** for electrical substations.

The Satlas path is deliberately conservative:

1. use only high-resolution SatlasPretrain tiles where `power_substation` was **explicitly annotated**;
2. convert Satlas' 8192×8192 zoom-13 polygons into the corresponding 512×512 NAIP zoom-17 chips;
3. keep true hard negatives only when `power_substation` is present but empty (or a child chip in an explicitly annotated parent contains no substation);
4. use Satlas as bootstrap/replay data, while Grid Mapper feedback and African held-out sets remain the main domain for safe promotion;
5. train RF-DETR with the aerial-imagery augmentation preset, evaluate fixed PR/F1 and per-geography F1, and export an ONNX package for QGIS.

The notebook still supports the original thesis/Roboflow data, TFOD folders, older VOC chips and immutable QGIS feedback snapshots.

**Use a GPU runtime in Colab. Training remains outside QGIS.**

> Satlas raw data is distributed in large archives. The converter copies only selected `power_substation` chips into the RF-DETR dataset, but it cannot make the upstream archive download itself smaller. Start with already-extracted Satlas data or the small NAIP sample, then add only the NAIP year archives you actually need.


In [ ]:
# 1 ── Install ───────────────────────────────────────────────────────────────
%pip -q install "rfdetr[train]" roboflow onnx onnxruntime supervision
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none – switch the runtime to GPU!")


In [ ]:
# 2 ── Settings ──────────────────────────────────────────────────────────────
TASK = "detect"          # "detect" (boxes) or "segment" (footprints – Satlas polygons support both)
MODEL_SIZE = "small"     # "nano" | "small" | "medium" | "large"; Small is a good first aerial model
EPOCHS = 60
BATCH_SIZE = 8           # lower to 4/2 if CUDA runs out of memory
GRAD_ACCUM = 2
LR = 1e-4
PRETRAINED = True         # RF-DETR's normal pretrained initialization
USE_AERIAL_AUGMENTATION = True
SCALE_JITTER = False      # avoids random crop clipping objects near aerial-chip borders

# SatlasPretrain bootstrap ----------------------------------------------------
# Expected extracted structure: SATLAS_ROOT/static, metadata, and naip (or naip_small).
# Set USE_SATLAS=True after the Satlas folders are available.
USE_SATLAS = False
SATLAS_ROOT = "/content/drive/MyDrive/GridMapper/SatlasPretrain"
SATLAS_NEGATIVE_RATIO = 2.0      # hard-negative chips per positive chip
SATLAS_MAX_POSITIVE_CHIPS = 4000
SATLAS_MAX_NEGATIVE_CHIPS = 8000
SATLAS_VALID_PCT = 10.0
# Keep Satlas' US/NAIP official test split out of the promotion test by default.
# The promotion gate should primarily be driven by your African fixed test data.
SATLAS_INCLUDE_OFFICIAL_TEST = False

# Optional download helpers. OFF by default because official Satlas archives can be large.
SATLAS_AUTO_DOWNLOAD_LABELS_METADATA = False
SATLAS_DOWNLOAD_SMALL_NAIP = False
SATLAS_NAIP_YEARS_TO_DOWNLOAD = []   # e.g. [2019, 2020]; each year archive is large

# Continual-learning settings -------------------------------------------------
GRIDMAPPER_SNAPSHOT_DIRS = [
    # "/content/drive/MyDrive/GridMapper/feedback/snapshots/feedback_20260922",
]
RESUME_CHECKPOINT = ""  # optional previous Grid Mapper RF-DETR .pth checkpoint

# Existing/baseline sources – keep them enabled on later rounds as replay data.
USE_ROBOFLOW = True
RF_WORKSPACE, RF_PROJECT, RF_VERSION = "substations", "ss-2", 1
ROBOFLOW_API_KEY = ""     # preferably use Colab secret ROBOFLOW_API_KEY
TFOD_DIRS = [
    # "/content/drive/MyDrive/01 Thesis & Research/Thesis/5_Roboflow 2.0 Object detection model",
]
VOC_CHIP_DIRS = [
    # "/content/drive/MyDrive/GridMapper/chips_zambia",
]
CLASS_NAME = "substation"

# Evaluation/promotion --------------------------------------------------------
PRED_THRESHOLD = 0.50
IOU_THRESHOLD = 0.50
MODEL_ID = ""             # blank -> timestamped id

WORK = "/content/work"
DRIVE_OUT = "/content/drive/MyDrive/GridMapper/models"
SEED = 42


In [ ]:
# Google Drive (needed for Drive data sources and to save the model)
from google.colab import drive
drive.mount("/content/drive")
if USE_ROBOFLOW and not ROBOFLOW_API_KEY:
    from google.colab import userdata
    try:
        ROBOFLOW_API_KEY = userdata.get("ROBOFLOW_API_KEY")
    except Exception:
        raise SystemExit("Add your Roboflow key as a Colab secret named ROBOFLOW_API_KEY, or set USE_ROBOFLOW = False")


In [ ]:
# 2a ── SatlasPretrain converter (self-contained) ───────────────────────────
# The same helper is shipped as scripts/satlas_bootstrap.py for VS Code use.
"""SatlasPretrain -> Grid Mapper RF-DETR bootstrap converter.

The converter intentionally ingests only ``power_substation`` annotations from the
high-resolution SatlasPretrain static labels and copies only NAIP 512x512 chips
that are actually selected for training/validation/test.

Satlas raw storage is distributed in large tar archives, so this module cannot
avoid downloading whichever source archive(s) the user chooses to obtain.  Once
those archives are extracted, however, it does *not* copy the rest of Satlas into
the Grid Mapper training dataset.

No QGIS dependency.  It is safe to run in Colab or a normal Python environment.
"""
from __future__ import annotations

import argparse
import hashlib
import json
import math
import os
import random
import shutil
from collections import Counter, defaultdict
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, Iterator, List, Optional, Sequence, Tuple

from PIL import Image

SATLAS_PARENT_PIXELS = 8192
SATLAS_CHILD_PIXELS = 512
SATLAS_CHILD_FACTOR = SATLAS_PARENT_PIXELS // SATLAS_CHILD_PIXELS  # 16 (z13 -> z17)
DEFAULT_CLASS = "power_substation"


def _stable_fraction(key: str) -> float:
    return int(hashlib.sha1(str(key).encode("utf-8")).hexdigest()[:8], 16) / 0xFFFFFFFF


def _tile_key(col: int, row: int) -> str:
    return f"{int(col)}_{int(row)}"


def _load_split(path: Path) -> set[Tuple[int, int]]:
    if not path.is_file():
        return set()
    raw = json.loads(path.read_text(encoding="utf-8"))
    return {(int(x[0]), int(x[1])) for x in raw}


def _clip_edge(points: Sequence[Tuple[float, float]], inside, intersect):
    if not points:
        return []
    out = []
    prev = points[-1]
    prev_in = inside(prev)
    for cur in points:
        cur_in = inside(cur)
        if cur_in:
            if not prev_in:
                out.append(intersect(prev, cur))
            out.append(cur)
        elif prev_in:
            out.append(intersect(prev, cur))
        prev, prev_in = cur, cur_in
    return out


def clip_polygon_to_rect(points: Sequence[Tuple[float, float]], xmin: float, ymin: float,
                         xmax: float, ymax: float) -> List[Tuple[float, float]]:
    """Sutherland-Hodgman clipping for one exterior polygon ring."""
    pts = [(float(x), float(y)) for x, y in points]
    if len(pts) >= 2 and pts[0] == pts[-1]:
        pts = pts[:-1]
    if len(pts) < 3:
        return []

    def ix(xval):
        def f(a, b):
            ax, ay = a; bx, by = b
            if bx == ax:
                return (xval, ay)
            t = (xval - ax) / (bx - ax)
            return (xval, ay + t * (by - ay))
        return f

    def iy(yval):
        def f(a, b):
            ax, ay = a; bx, by = b
            if by == ay:
                return (ax, yval)
            t = (yval - ay) / (by - ay)
            return (ax + t * (bx - ax), yval)
        return f

    pts = _clip_edge(pts, lambda p: p[0] >= xmin, ix(xmin))
    pts = _clip_edge(pts, lambda p: p[0] <= xmax, ix(xmax))
    pts = _clip_edge(pts, lambda p: p[1] >= ymin, iy(ymin))
    pts = _clip_edge(pts, lambda p: p[1] <= ymax, iy(ymax))
    # Drop consecutive duplicates caused by clipping.
    cleaned = []
    for p in pts:
        if not cleaned or abs(p[0] - cleaned[-1][0]) > 1e-6 or abs(p[1] - cleaned[-1][1]) > 1e-6:
            cleaned.append(p)
    if len(cleaned) >= 2 and cleaned[0] == cleaned[-1]:
        cleaned.pop()
    return cleaned if len(cleaned) >= 3 else []


def _polygon_area(points: Sequence[Tuple[float, float]]) -> float:
    if len(points) < 3:
        return 0.0
    return abs(sum(points[i][0] * points[(i + 1) % len(points)][1] -
                   points[(i + 1) % len(points)][0] * points[i][1]
                   for i in range(len(points))) / 2.0)


def _bbox(points: Sequence[Tuple[float, float]]) -> Tuple[float, float, float, float]:
    xs = [p[0] for p in points]; ys = [p[1] for p in points]
    return min(xs), min(ys), max(xs), max(ys)


def _image_name_from_vector(data: dict) -> Optional[str]:
    meta = data.get("metadata") or {}
    for key in ("ImageName", "image_name", "imagename"):
        if meta.get(key):
            return str(meta[key])
    return None


class NaipLocator:
    """Find high-resolution NAIP tiles, preferring Satlas' ImageName metadata."""
    def __init__(self, satlas_root: os.PathLike):
        self.root = Path(satlas_root)
        self.naip = self.root / "naip"
        if not self.naip.exists() and (self.root / "naip_small").exists():
            self.naip = self.root / "naip_small"
        self.cache: Dict[Tuple[Optional[str], int, int], Optional[Path]] = {}

    def find(self, image_name: Optional[str], col17: int, row17: int) -> Optional[Path]:
        key = (image_name, int(col17), int(row17))
        if key in self.cache:
            return self.cache[key]
        fname = f"{col17}_{row17}.png"
        if image_name:
            p = self.naip / image_name / "tci" / fname
            if p.is_file():
                self.cache[key] = p
                return p
        # Fallback for static labels without usable metadata or partial NAIP roots.
        matches = sorted(self.naip.glob(f"*/tci/{fname}")) if self.naip.exists() else []
        p = matches[-1] if matches else None
        self.cache[key] = p
        return p


@dataclass
class SatlasSample:
    image_path: Path
    split: str
    parent_tile: Tuple[int, int]
    child_tile: Tuple[int, int]
    polygons: List[List[float]]
    image_name: Optional[str]
    positive: bool

    @property
    def stable_key(self) -> str:
        return f"satlas:{self.parent_tile[0]}:{self.parent_tile[1]}:{self.child_tile[0]}:{self.child_tile[1]}"


def iter_satlas_samples(satlas_root: os.PathLike, class_name: str = DEFAULT_CLASS,
                        negative_ratio: float = 2.0, max_positive_chips: int = 4000,
                        max_negative_chips: int = 8000, valid_pct: float = 10.0,
                        include_official_test: bool = False,
                        min_polygon_area_px: float = 16.0) -> Iterator[SatlasSample]:
    """Yield selected high-resolution Satlas chips with substation labels.

    Only static label folders where ``class_name`` is explicitly present are
    considered.  This is important: a missing class key means the class was not
    annotated, while ``"power_substation": []`` is a valid hard negative.

    Satlas' zoom-13 8192x8192 annotation coordinates are clipped into the
    corresponding 16x16 zoom-17 NAIP chips (512x512 each).
    """
    root = Path(satlas_root)
    static_root = root / "static"
    meta_root = root / "metadata"
    if not static_root.is_dir():
        raise FileNotFoundError(f"Satlas static labels not found: {static_root}")
    locator = NaipLocator(root)
    if not locator.naip.is_dir():
        raise FileNotFoundError(f"Satlas NAIP imagery not found under {root}/naip or {root}/naip_small")

    train_tiles = _load_split(meta_root / "train_highres.json")
    test_tiles = _load_split(meta_root / "test_highres.json")

    positives: List[SatlasSample] = []
    negative_candidates: List[SatlasSample] = []

    for vf in sorted(static_root.glob("*/vector.json")):
        try:
            col13, row13 = (int(x) for x in vf.parent.name.split("_")[:2])
        except Exception:
            continue
        data = json.loads(vf.read_text(encoding="utf-8"))
        if class_name not in data:
            continue  # not annotated for this class; never assume it is negative
        parent = (col13, row13)
        if parent in test_tiles and not include_official_test:
            continue
        if parent in test_tiles:
            split = "test"
        else:
            # Preserve official training domain, then carve a deterministic validation set.
            frac = _stable_fraction(f"satlas-valid:{col13}:{row13}") * 100.0
            split = "valid" if frac < float(valid_pct) else "train"

        image_name = _image_name_from_vector(data)
        per_child: Dict[Tuple[int, int], List[List[float]]] = defaultdict(list)
        features = data.get(class_name) or []
        for feat in features:
            geom = (feat or {}).get("Geometry") or {}
            if str(geom.get("Type", "")).lower() != "polygon":
                continue
            rings = geom.get("Polygon") or []
            if not rings:
                continue
            outer = [(float(x), float(y)) for x, y in rings[0]]
            if len(outer) < 3:
                continue
            bx1, by1, bx2, by2 = _bbox(outer)
            sx0 = max(0, min(15, int(math.floor(bx1 / SATLAS_CHILD_PIXELS))))
            sy0 = max(0, min(15, int(math.floor(by1 / SATLAS_CHILD_PIXELS))))
            sx1 = max(0, min(15, int(math.floor(max(0.0, bx2 - 1e-9) / SATLAS_CHILD_PIXELS))))
            sy1 = max(0, min(15, int(math.floor(max(0.0, by2 - 1e-9) / SATLAS_CHILD_PIXELS))))
            for sy in range(sy0, sy1 + 1):
                for sx in range(sx0, sx1 + 1):
                    xmin, ymin = sx * SATLAS_CHILD_PIXELS, sy * SATLAS_CHILD_PIXELS
                    clipped = clip_polygon_to_rect(outer, xmin, ymin,
                                                   xmin + SATLAS_CHILD_PIXELS,
                                                   ymin + SATLAS_CHILD_PIXELS)
                    if len(clipped) < 3 or _polygon_area(clipped) < min_polygon_area_px:
                        continue
                    local = []
                    for x, y in clipped:
                        local.extend([min(512.0, max(0.0, x - xmin)),
                                      min(512.0, max(0.0, y - ymin))])
                    per_child[(sx, sy)].append(local)

        # Positive child chips.
        for (sx, sy), polygons in per_child.items():
            c17, r17 = col13 * SATLAS_CHILD_FACTOR + sx, row13 * SATLAS_CHILD_FACTOR + sy
            img = locator.find(image_name, c17, r17)
            if img:
                positives.append(SatlasSample(img, split, parent, (c17, r17), polygons, image_name, True))

        # Any child chip in an explicitly annotated parent tile that contains no
        # substation polygon is a valid negative candidate.
        positive_children = set(per_child)
        for sy in range(SATLAS_CHILD_FACTOR):
            for sx in range(SATLAS_CHILD_FACTOR):
                if (sx, sy) in positive_children:
                    continue
                c17, r17 = col13 * SATLAS_CHILD_FACTOR + sx, row13 * SATLAS_CHILD_FACTOR + sy
                img = locator.find(image_name, c17, r17)
                if img:
                    negative_candidates.append(SatlasSample(img, split, parent, (c17, r17), [], image_name, False))

    # Deterministic cap so repeated training runs see the same bootstrap set.
    positives.sort(key=lambda s: hashlib.sha1(s.stable_key.encode()).hexdigest())
    if max_positive_chips and len(positives) > max_positive_chips:
        positives = positives[:max_positive_chips]

    wanted_neg = int(round(len(positives) * max(0.0, negative_ratio)))
    if max_negative_chips:
        wanted_neg = min(wanted_neg, int(max_negative_chips))
    negative_candidates.sort(key=lambda s: hashlib.sha1(("neg:" + s.stable_key).encode()).hexdigest())
    negatives = negative_candidates[:wanted_neg]

    # Stable ordering across positives/negatives.
    combined = positives + negatives
    combined.sort(key=lambda s: hashlib.sha1((s.split + ":" + s.stable_key).encode()).hexdigest())
    yield from combined


def selection_summary(samples: Sequence[SatlasSample]) -> dict:
    by_split = Counter(s.split for s in samples)
    pos_split = Counter(s.split for s in samples if s.positive)
    neg_split = Counter(s.split for s in samples if not s.positive)
    return {
        "samples": len(samples),
        "positive_chips": sum(s.positive for s in samples),
        "negative_chips": sum(not s.positive for s in samples),
        "by_split": dict(by_split),
        "positive_by_split": dict(pos_split),
        "negative_by_split": dict(neg_split),
    }


def export_coco(samples: Sequence[SatlasSample], output_dir: os.PathLike,
                copy_mode: str = "copy", class_display_name: str = "substation") -> dict:
    """Export selected Satlas samples as an RF-DETR compatible COCO dataset."""
    out = Path(output_dir)
    if out.exists():
        shutil.rmtree(out)
    stores = {s: {"images": [], "annotations": []} for s in ("train", "valid", "test")}
    img_id = ann_id = 0
    for sample in samples:
        split = sample.split
        split_dir = out / split
        split_dir.mkdir(parents=True, exist_ok=True)
        img_id += 1
        dst_name = f"satlas_{sample.parent_tile[0]}_{sample.parent_tile[1]}_{sample.child_tile[0]}_{sample.child_tile[1]}.jpg"
        dst = split_dir / dst_name
        with Image.open(sample.image_path) as im:
            im = im.convert("RGB")
            w, h = im.size
            im.save(dst, quality=95)
        stores[split]["images"].append({
            "id": img_id, "file_name": dst_name, "width": w, "height": h,
            "gridmapper_source": "satlaspretrain_power_substation",
            "gridmapper_geography": "satlas_naip",
            "gridmapper_satlas_parent": _tile_key(*sample.parent_tile),
            "gridmapper_satlas_child": _tile_key(*sample.child_tile),
        })
        for poly in sample.polygons:
            xs, ys = poly[0::2], poly[1::2]
            x1, y1, x2, y2 = min(xs), min(ys), max(xs), max(ys)
            if x2 - x1 < 2 or y2 - y1 < 2:
                continue
            ann_id += 1
            stores[split]["annotations"].append({
                "id": ann_id, "image_id": img_id, "category_id": 1, "iscrowd": 0,
                "bbox": [x1, y1, x2 - x1, y2 - y1], "area": (x2 - x1) * (y2 - y1),
                "segmentation": [list(map(float, poly))],
            })

    cats = [{"id": 0, "name": class_display_name + "s", "supercategory": "none"},
            {"id": 1, "name": class_display_name, "supercategory": class_display_name + "s"}]
    for split, store in stores.items():
        (out / split).mkdir(parents=True, exist_ok=True)
        (out / split / "_annotations.coco.json").write_text(json.dumps({
            "images": store["images"], "annotations": store["annotations"], "categories": cats,
            "info": {
                "description": "Grid Mapper SatlasPretrain power_substation bootstrap",
                "source": "AllenAI SatlasPretrain high-resolution NAIP",
                "license_notice": "Satlas contains mixed-source labels; preserve source attribution and review SatlasPretrain source licenses.",
            },
        }, indent=2), encoding="utf-8")
    manifest = selection_summary(samples)
    manifest.update({
        "class": DEFAULT_CLASS,
        "source": "SatlasPretrain",
        "notes": [
            "Only tiles explicitly annotated for power_substation were used.",
            "An absent power_substation key was never treated as a negative.",
            "NAIP imagery is public domain; Satlas annotations include mixed source licenses including ODbL and ODC-BY.",
        ],
    })
    (out / "satlas_bootstrap_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    return manifest


def scan_label_catalog(satlas_root: os.PathLike, class_name: str = DEFAULT_CLASS) -> dict:
    """Scan labels without needing imagery; useful before downloading NAIP archives."""
    root = Path(satlas_root)
    static_root = root / "static"
    result = {"annotated_tiles": 0, "positive_parent_tiles": 0, "negative_parent_tiles": 0,
              "instances": 0, "image_names": [], "years": {}}
    image_names = set(); years = Counter()
    for vf in sorted(static_root.glob("*/vector.json")):
        data = json.loads(vf.read_text(encoding="utf-8"))
        if class_name not in data:
            continue
        result["annotated_tiles"] += 1
        feats = data.get(class_name) or []
        if feats:
            result["positive_parent_tiles"] += 1
            result["instances"] += len(feats)
        else:
            result["negative_parent_tiles"] += 1
        name = _image_name_from_vector(data)
        if name:
            image_names.add(name)
            tail = name.rsplit("_", 1)[-1]
            year = tail[:4] if len(tail) >= 4 and tail[:4].isdigit() else "unknown"
            years[year] += 1
    result["image_names"] = sorted(image_names)
    result["years"] = dict(sorted(years.items()))
    return result


def main(argv=None):
    ap = argparse.ArgumentParser(description=__doc__)
    ap.add_argument("--satlas-root", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--negative-ratio", type=float, default=2.0)
    ap.add_argument("--max-positive", type=int, default=4000)
    ap.add_argument("--max-negative", type=int, default=8000)
    ap.add_argument("--valid-pct", type=float, default=10.0)
    ap.add_argument("--include-official-test", action="store_true")
    ap.add_argument("--catalog-only", action="store_true")
    args = ap.parse_args(argv)
    if args.catalog_only:
        summary = scan_label_catalog(args.satlas_root)
        Path(args.out).write_text(json.dumps(summary, indent=2), encoding="utf-8")
        print(json.dumps({k: v for k, v in summary.items() if k != "image_names"}, indent=2))
        return 0
    samples = list(iter_satlas_samples(
        args.satlas_root, negative_ratio=args.negative_ratio,
        max_positive_chips=args.max_positive, max_negative_chips=args.max_negative,
        valid_pct=args.valid_pct, include_official_test=args.include_official_test,
    ))
    manifest = export_coco(samples, args.out)
    print(json.dumps(manifest, indent=2))
    return 0



In [ ]:
# 2b ── Optional Satlas download + catalog scan ──────────────────────────────
# Official source archives.  We intentionally do NOT download them unless you opt in.
# Raw Satlas is archive-based, so network transfer happens by archive; after extraction,
# Grid Mapper only ingests the selected power_substation chips.
import os, pathlib, subprocess, tarfile, json

SATLAS_URLS = {
    "labels_static": "https://ai2-public-datasets.s3.amazonaws.com/satlas/satlas-dataset-v1-labels-static.tar",
    "metadata": "https://ai2-public-datasets.s3.amazonaws.com/satlas/satlas-dataset-v1-metadata.tar",
    "naip_small": "https://ai2-public-datasets.s3.amazonaws.com/satlas/satlas-dataset-v1-naip-small.tar",
}

def _download_extract(url, dest):
    dest = pathlib.Path(dest); dest.mkdir(parents=True, exist_ok=True)
    archive = dest / pathlib.Path(url).name
    if not archive.exists():
        print("Downloading", url)
        subprocess.run(["wget", "-c", url, "-O", str(archive)], check=True)
    print("Extracting", archive.name)
    with tarfile.open(archive, "r:*") as tf:
        tf.extractall(dest)

if USE_SATLAS and SATLAS_AUTO_DOWNLOAD_LABELS_METADATA:
    _download_extract(SATLAS_URLS["labels_static"], SATLAS_ROOT)
    _download_extract(SATLAS_URLS["metadata"], SATLAS_ROOT)
if USE_SATLAS and SATLAS_DOWNLOAD_SMALL_NAIP:
    _download_extract(SATLAS_URLS["naip_small"], SATLAS_ROOT)
    # Satlas docs suggest symlinking naip_small -> naip; the converter accepts either name.
if USE_SATLAS and SATLAS_NAIP_YEARS_TO_DOWNLOAD:
    for year in SATLAS_NAIP_YEARS_TO_DOWNLOAD:
        _download_extract(
            f"https://ai2-public-datasets.s3.amazonaws.com/satlas/satlas-dataset-v1-naip-{int(year)}.tar",
            SATLAS_ROOT,
        )

if USE_SATLAS:
    static_dir = pathlib.Path(SATLAS_ROOT) / "static"
    if static_dir.is_dir():
        cat = scan_label_catalog(SATLAS_ROOT)
        compact = {k:v for k,v in cat.items() if k != "image_names"}
        print("Satlas power_substation label catalog:")
        print(json.dumps(compact, indent=2))
        print("Unique referenced image names:", len(cat.get("image_names", [])))
        if cat.get("years"):
            print("Referenced NAIP years:", cat["years"])
    else:
        raise SystemExit(
            f"USE_SATLAS=True but {static_dir} does not exist. Extract Satlas labels/metadata first, "
            "or enable SATLAS_AUTO_DOWNLOAD_LABELS_METADATA."
        )


In [ ]:
# 3 ── Build one replay dataset (Satlas + African data + feedback) ─────────
import csv, glob, hashlib, json, os, shutil
import xml.etree.ElementTree as ET
from PIL import Image

DATASET = os.path.join(WORK, "dataset")
SPLITS = ("train", "valid", "test")
shutil.rmtree(DATASET, ignore_errors=True)
store = {s: {"images": [], "annotations": []} for s in SPLITS}
counter = {"img": 0, "ann": 0}
CAT_ID = 1


def add_image(split, src_path, boxes=(), polygons=(), metadata=None):
    """Copy one image into the combined dataset and merge its labels."""
    metadata = metadata or {}
    os.makedirs(os.path.join(DATASET, split), exist_ok=True)
    counter["img"] += 1
    iid = counter["img"]
    name = f"{iid:07d}_{os.path.basename(src_path)}"
    dst = os.path.join(DATASET, split, name)
    with Image.open(src_path) as im:
        im = im.convert("RGB")
        w, h = im.size
        im.save(dst, quality=95)
    image_row = {"id": iid, "file_name": name, "width": w, "height": h}
    # COCO readers ignore unknown fields; Grid Mapper uses these for geography F1.
    for k, v in metadata.items():
        if str(k).startswith("gridmapper_"):
            image_row[k] = v
    store[split]["images"].append(image_row)
    items = [(p, None) for p in polygons] + [(None, b) for b in boxes]
    for poly, box in items:
        if poly is not None:
            xs, ys = poly[0::2], poly[1::2]
            box = (min(xs), min(ys), max(xs), max(ys))
        x1, y1, x2, y2 = [float(v) for v in box]
        x1, y1, x2, y2 = max(0, x1), max(0, y1), min(w, x2), min(h, y2)
        if x2 - x1 < 2 or y2 - y1 < 2:
            continue
        counter["ann"] += 1
        store[split]["annotations"].append({
            "id": counter["ann"], "image_id": iid, "category_id": CAT_ID, "iscrowd": 0,
            "bbox": [x1, y1, x2 - x1, y2 - y1], "area": (x2 - x1) * (y2 - y1),
            "segmentation": [list(map(float, poly))] if poly is not None else [],
        })


def stable_split_for_key(key, valid_pct=10, test_pct=10):
    """Hash split: additions never reshuffle previously seen files."""
    bucket = int(hashlib.sha1(str(key).encode()).hexdigest()[:8], 16) % 10000 / 100
    if bucket < test_pct:
        return "test"
    if bucket < test_pct + valid_pct:
        return "valid"
    return "train"



# 3a SatlasPretrain high-resolution bootstrap --------------------------------
satlas_samples = []
if USE_SATLAS:
    print("Selecting SatlasPretrain power_substation chips from:", SATLAS_ROOT)
    satlas_samples = list(iter_satlas_samples(
        SATLAS_ROOT,
        class_name="power_substation",
        negative_ratio=SATLAS_NEGATIVE_RATIO,
        max_positive_chips=SATLAS_MAX_POSITIVE_CHIPS,
        max_negative_chips=SATLAS_MAX_NEGATIVE_CHIPS,
        valid_pct=SATLAS_VALID_PCT,
        include_official_test=SATLAS_INCLUDE_OFFICIAL_TEST,
    ))
    print("Satlas selection:", selection_summary(satlas_samples))
    if not satlas_samples:
        print("WARNING: no matching Satlas imagery chips were found. Labels may exist but the required NAIP archives may not be extracted.")
    for sample in satlas_samples:
        if TASK == "segment":
            boxes, polys = [], sample.polygons
        else:
            boxes, polys = [], []
            for p in sample.polygons:
                xs, ys = p[0::2], p[1::2]
                boxes.append((min(xs), min(ys), max(xs), max(ys)))
        add_image(sample.split, str(sample.image_path), boxes, polys, {
            "gridmapper_source": "satlaspretrain_power_substation",
            "gridmapper_geography": "satlas_naip",
            "gridmapper_satlas_parent": f"{sample.parent_tile[0]}_{sample.parent_tile[1]}",
            "gridmapper_satlas_child": f"{sample.child_tile[0]}_{sample.child_tile[1]}",
        })

# 3b Roboflow baseline (keeps its provider split)
if USE_ROBOFLOW:
    from roboflow import Roboflow
    fmt = "coco-segmentation" if TASK == "segment" else "coco"
    rf_dir = os.path.join(WORK, "roboflow")
    Roboflow(api_key=ROBOFLOW_API_KEY).workspace(RF_WORKSPACE).project(RF_PROJECT) \
        .version(RF_VERSION).download(fmt, location=rf_dir, overwrite=True)
    for split in SPLITS:
        ann_path = os.path.join(rf_dir, split, "_annotations.coco.json")
        if not os.path.isfile(ann_path):
            continue
        coco = json.load(open(ann_path))
        per_img = {}
        for a in coco["annotations"]:
            per_img.setdefault(a["image_id"], []).append(a)
        for img in coco["images"]:
            boxes, polys = [], []
            for a in per_img.get(img["id"], []):
                seg = a.get("segmentation")
                if TASK == "segment" and isinstance(seg, list) and seg and len(seg[0]) >= 6:
                    polys.append(seg[0])
                else:
                    x, y, bw, bh = a["bbox"]
                    boxes.append((x, y, x + bw, y + bh))
            add_image(split, os.path.join(rf_dir, split, img["file_name"]), boxes, polys,
                      {"gridmapper_source": "roboflow_baseline", "gridmapper_geography": "baseline"})

# 3c TensorFlow Object Detection CSV folders
for root in TFOD_DIRS:
    for split in SPLITS:
        csv_path = os.path.join(root, split, "_annotations.csv")
        if not os.path.isfile(csv_path):
            continue
        rows = {}
        for r in csv.DictReader(open(csv_path)):
            rows.setdefault(r["filename"], []).append(
                (float(r["xmin"]), float(r["ymin"]), float(r["xmax"]), float(r["ymax"])))
        for fname, boxes in rows.items():
            p = os.path.join(root, split, fname)
            if os.path.isfile(p):
                add_image(split, p, boxes, metadata={"gridmapper_source": "tfod_baseline",
                                                     "gridmapper_geography": "baseline"})

# 3d Older Grid Mapper Pascal-VOC chips – stable, filename-based split
for root in VOC_CHIP_DIRS:
    xmls = sorted(x for x in glob.glob(os.path.join(root, "*.xml")) if not x.endswith(".aux.xml"))
    for x in xmls:
        t = ET.parse(x).getroot()
        if t.tag != "annotation" or not t.findtext("filename"):
            continue
        img = os.path.join(root, t.findtext("filename"))
        boxes = [tuple(float(o.find("bndbox").findtext(k)) for k in ("xmin", "ymin", "xmax", "ymax"))
                 for o in t.findall("object")]
        if os.path.isfile(img):
            add_image(stable_split_for_key(os.path.abspath(img)), img, boxes,
                      metadata={"gridmapper_source": "voc_baseline", "gridmapper_geography": "baseline"})

# 3e Human-reviewed Grid Mapper snapshots – preserve their FIXED split exactly.
for root in GRIDMAPPER_SNAPSHOT_DIRS:
    print("Loading feedback snapshot:", root)
    for split in SPLITS:
        ann_path = os.path.join(root, split, "_annotations.coco.json")
        if not os.path.isfile(ann_path):
            continue
        coco = json.load(open(ann_path))
        per_img = {}
        for a in coco.get("annotations", []):
            per_img.setdefault(a["image_id"], []).append(a)
        for img in coco.get("images", []):
            boxes, polys = [], []
            for a in per_img.get(img["id"], []):
                seg = a.get("segmentation")
                if TASK == "segment" and isinstance(seg, list) and seg and len(seg[0]) >= 6:
                    polys.append(seg[0])
                else:
                    x, y, bw, bh = a["bbox"]
                    boxes.append((x, y, x + bw, y + bh))
            meta = {k: v for k, v in img.items() if str(k).startswith("gridmapper_")}
            meta.setdefault("gridmapper_source", "feedback_snapshot")
            p = os.path.join(root, split, img["file_name"])
            if os.path.isfile(p):
                # Empty boxes/polygons are intentional hard-negative examples.
                add_image(split, p, boxes, polys, meta)

categories = [{"id": 0, "name": CLASS_NAME + "s", "supercategory": "none"},
              {"id": CAT_ID, "name": CLASS_NAME, "supercategory": CLASS_NAME + "s"}]
for s in SPLITS:
    os.makedirs(os.path.join(DATASET, s), exist_ok=True)
    json.dump({"images": store[s]["images"], "annotations": store[s]["annotations"],
               "categories": categories,
               "info": {"description": "Grid Mapper Satlas bootstrap + African replay + human feedback dataset",
                        "fixed_feedback_splits": True}},
              open(os.path.join(DATASET, s, "_annotations.coco.json"), "w"), indent=2)
    negatives = len(store[s]["images"]) - len({a["image_id"] for a in store[s]["annotations"]})
    print(f"{s:5s}: {len(store[s]['images']):4d} images, {len(store[s]['annotations']):4d} substations, "
          f"{negatives:4d} negative images")

assert store["train"]["annotations"], "No training labels found – check data source settings"
n_polys = sum(1 for s in SPLITS for a in store[s]["annotations"] if a["segmentation"])
if TASK == "segment" and n_polys == 0:
    raise SystemExit('TASK="segment" needs polygon labels. Use detection mode until polygon feedback is available.')


In [ ]:
# 4 ── Train / fine-tune ────────────────────────────────────────────────────
import rfdetr
from rfdetr import RFDETR
from rfdetr.datasets.aug_configs import AUG_AERIAL

VARIANTS = {
    ("detect", "nano"): "RFDETRNano", ("detect", "small"): "RFDETRSmall",
    ("detect", "medium"): "RFDETRMedium", ("detect", "large"): "RFDETRLarge",
    ("segment", "nano"): "RFDETRSegNano", ("segment", "small"): "RFDETRSegSmall",
    ("segment", "medium"): "RFDETRSegMedium", ("segment", "large"): "RFDETRSegLarge",
}
ModelClass = getattr(rfdetr, VARIANTS[(TASK, MODEL_SIZE)])
if RESUME_CHECKPOINT and os.path.isfile(RESUME_CHECKPOINT):
    print("Warm-starting from:", RESUME_CHECKPOINT)
    model = RFDETR.from_checkpoint(RESUME_CHECKPOINT, trust_checkpoint=True)
else:
    model = ModelClass() if PRETRAINED else ModelClass(pretrain_weights=None)

RUN_DIR = os.path.join(WORK, f"run_{TASK}_{MODEL_SIZE}")
train_kwargs = dict(dataset_dir=DATASET, epochs=EPOCHS, batch_size=BATCH_SIZE,
                    grad_accum_steps=GRAD_ACCUM, lr=LR, output_dir=RUN_DIR,
                    early_stopping=True, early_stopping_patience=10, scale_jitter=SCALE_JITTER)
if USE_AERIAL_AUGMENTATION:
    train_kwargs["aug_config"] = AUG_AERIAL
    print("Using RF-DETR AUG_AERIAL augmentation preset")
model.train(**train_kwargs)

best = os.path.join(RUN_DIR, "checkpoint_best_total.pth")
if os.path.isfile(best):
    model = RFDETR.from_checkpoint(best, trust_checkpoint=True)
print("Model ready:", type(model).__name__)
print("Best checkpoint:", best if os.path.isfile(best) else "not found")


In [ ]:
# 5 ── Evaluate globally AND by geography ──────────────────────────────────
import numpy as np
import supervision as sv
from IPython.display import display

metrics = {}
try:
    raw_metrics = model.evaluate(split="test", dataset_dir=DATASET, batch_size=BATCH_SIZE, output_dir=RUN_DIR)
    metrics.update({str(k): float(v) for k, v in raw_metrics.items() if isinstance(v, (int, float))})
    print("RF-DETR metrics:", {k: round(v, 4) for k, v in metrics.items()})
except Exception as e:
    print("Built-in evaluation skipped:", e)


def iou(a, b):
    x1, y1 = max(a[0], b[0]), max(a[1], b[1])
    x2, y2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, x2-x1) * max(0, y2-y1)
    if inter <= 0: return 0.0
    aa = max(0, a[2]-a[0]) * max(0, a[3]-a[1])
    bb = max(0, b[2]-b[0]) * max(0, b[3]-b[1])
    return inter / max(aa + bb - inter, 1e-9)


def prf(tp, fp, fn):
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    f = 2*p*r/(p+r) if p+r else 0.0
    return {"tp": int(tp), "fp": int(fp), "fn": int(fn),
            "precision": p, "recall": r, "f1": f}


def evaluate_fixed_test():
    anns_by_img = {}
    for a in store["test"]["annotations"]:
        x, y, w, h = a["bbox"]
        anns_by_img.setdefault(a["image_id"], []).append([x, y, x+w, y+h])
    totals = [0, 0, 0]
    geo = {}
    for img in store["test"]["images"]:
        path = os.path.join(DATASET, "test", img["file_name"])
        det = model.predict(Image.open(path).convert("RGB"), threshold=PRED_THRESHOLD)
        preds = np.asarray(getattr(det, "xyxy", np.empty((0,4))), dtype=float).reshape(-1,4)
        conf = np.asarray(getattr(det, "confidence", np.ones(len(preds))), dtype=float)
        order = np.argsort(-conf) if len(conf) else np.arange(len(preds))
        gts = anns_by_img.get(img["id"], [])
        used = set(); tp = fp = 0
        for pi in order:
            best_i, best_j = 0.0, None
            for j, gt in enumerate(gts):
                if j in used: continue
                v = iou(preds[pi], gt)
                if v > best_i:
                    best_i, best_j = v, j
            if best_j is not None and best_i >= IOU_THRESHOLD:
                used.add(best_j); tp += 1
            else:
                fp += 1
        fn = len(gts) - len(used)
        totals[0] += tp; totals[1] += fp; totals[2] += fn
        name = str(img.get("gridmapper_geography") or "baseline")
        geo.setdefault(name, [0,0,0])
        geo[name][0] += tp; geo[name][1] += fp; geo[name][2] += fn
    return prf(*totals), {k: prf(*v) for k, v in geo.items()}

fixed, geography_metrics = evaluate_fixed_test()
metrics.update({k: float(v) for k, v in fixed.items() if k in ("precision", "recall", "f1")})
print("Fixed-test PR/F1:", {k: round(v, 4) if isinstance(v,float) else v for k,v in fixed.items()})
print("By geography:")
for geo, vals in geography_metrics.items():
    print(" ", geo, {k: round(v,4) if isinstance(v,float) else v for k,v in vals.items()})

# Visual sanity check
test_imgs = [os.path.join(DATASET, "test", i["file_name"]) for i in store["test"]["images"]][:6]
tiles = []
for p in test_imgs:
    im = Image.open(p).convert("RGB")
    det = model.predict(im, threshold=PRED_THRESHOLD)
    ann = sv.BoxAnnotator(thickness=3).annotate(im.copy(), det)
    if TASK == "segment" and getattr(det, "mask", None) is not None:
        ann = sv.MaskAnnotator().annotate(ann, det)
    tiles.append(ann.resize((320, 320)))
if tiles:
    grid = Image.new("RGB", (320 * min(3, len(tiles)), 320 * ((len(tiles) + 2) // 3)))
    for k, t in enumerate(tiles):
        grid.paste(t, ((k % 3) * 320, (k // 3) * 320))
    display(grid)


In [ ]:
# 6 ── Export candidate for QGIS (ONNX + promotion metadata) ───────────────
import datetime, zipfile
now_utc = lambda: datetime.datetime.now(datetime.timezone.utc)

EXPORT_DIR = os.path.join(WORK, "export")
shutil.rmtree(EXPORT_DIR, ignore_errors=True)
os.makedirs(EXPORT_DIR, exist_ok=True)
names = getattr(getattr(model, "model", None), "class_names", None) or getattr(model, "class_names", None)
if isinstance(names, dict):
    class_names = {str(k): v for k, v in names.items()}
elif names:
    class_names = {str(i): n for i, n in enumerate(names)}
else:
    class_names = {"0": CLASS_NAME}

stamp = now_utc().strftime("%Y%m%d_%H%M")
resolved_model_id = MODEL_ID.strip() or f"gridmapper_{TASK}_{MODEL_SIZE}_{stamp}"
info = {
    "model_id": resolved_model_id,
    "arch": "rfdetr", "task": TASK, "variant": VARIANTS[(TASK, MODEL_SIZE)],
    "class_names": class_names, "background_class_id": -1,
    "trained": now_utc().strftime("%Y-%m-%d %H:%M UTC"), "epochs": EPOCHS,
    "images": {s: len(store[s]["images"]) for s in SPLITS},
    "metrics": metrics,
    "primary_metric": "f1",
    "geography_metrics": geography_metrics,
    "prediction_threshold": PRED_THRESHOLD, "iou_threshold": IOU_THRESHOLD,
    "training_snapshots": list(GRIDMAPPER_SNAPSHOT_DIRS),
    "parent_checkpoint": RESUME_CHECKPOINT or None,
    "replay_baseline_enabled": bool(USE_SATLAS or USE_ROBOFLOW or TFOD_DIRS or VOC_CHIP_DIRS),
    "bootstrap_sources": {
        "satlaspretrain": {
            "enabled": bool(USE_SATLAS),
            "selected_chips": len(satlas_samples),
            "positive_chips": sum(1 for s in satlas_samples if s.positive),
            "negative_chips": sum(1 for s in satlas_samples if not s.positive),
            "class": "power_substation",
            "imagery": "NAIP 0.5-2 m/pixel high-resolution SatlasPretrain",
            "official_test_included": bool(SATLAS_INCLUDE_OFFICIAL_TEST),
        }
    },
    "licence": ("RF-DETR Apache-2.0. SatlasPretrain uses mixed-source data/labels; "
                "NAIP imagery is public domain and Satlas documentation lists source-specific licences "
                "including ODbL and ODC-BY. Other training imagery remains subject to provider terms."),
}
onnx_path = str(model.export(output_dir=EXPORT_DIR, notes=info, verbose=False,
                             output_name=f"gridmapper_{TASK}_{MODEL_SIZE}"))
json_path = os.path.splitext(onnx_path)[0] + ".json"
with open(json_path, "w") as fh:
    json.dump(info, fh, indent=2)

zip_path = os.path.join(WORK, resolved_model_id + ".zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(onnx_path, os.path.basename(onnx_path))
    z.write(json_path, os.path.basename(json_path))

# Quick ONNX Runtime check – the same CPU runtime used by the QGIS plugin.
import onnxruntime as ort
sess = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
inp = sess.get_inputs()[0]
outs = sess.run(None, {inp.name: np.zeros([d if isinstance(d, int) else 1 for d in inp.shape], np.float32)})
print("ONNX OK – input", inp.shape, "outputs", [o.name for o in sess.get_outputs()], [o.shape for o in outs])
print("Candidate package:", zip_path, f"({os.path.getsize(zip_path) / 1e6:.0f} MB)")
print("Primary F1:", round(metrics.get("f1", -1), 4))


In [ ]:
# 7 ── Save candidate AND training checkpoint to Drive ─────────────────────
os.makedirs(DRIVE_OUT, exist_ok=True)
model_dest = os.path.join(DRIVE_OUT, os.path.basename(zip_path))
shutil.copy2(zip_path, model_dest)
print("Candidate saved:", model_dest)

checkpoint_dest = None
if os.path.isfile(best):
    checkpoint_dest = os.path.join(DRIVE_OUT, resolved_model_id + "_checkpoint.pth")
    shutil.copy2(best, checkpoint_dest)
    print("Checkpoint saved for the next continual-learning round:", checkpoint_dest)

from google.colab import files
files.download(zip_path)


## Activate the candidate in QGIS

1. In QGIS run **Grid Mapper → Register / safely promote a candidate model**.
2. Select the exported `gridmapper_....zip`.
3. Grid Mapper reads `metrics`, `primary_metric = f1`, and `geography_metrics` from the package.
4. If the overall validation metric does not improve enough, or a shared geography regresses beyond the configured gate, the candidate is **registered but not activated**.
5. If it passes, it becomes the active model used by local inference and the country-scanning AI queue.

### Continual-learning rule

Keep the original thesis/Roboflow data and the selected Satlas bootstrap examples as replay data, then add each new immutable QGIS feedback snapshot. Do not train only on the newest detections. Keep African validation/test examples fixed so the model registry can detect geographic regressions.

### Satlas provenance

The notebook records Satlas as a distinct source in model metadata. SatlasPretrain combines datasets under several source-specific licences; preserve the attribution/provenance information and review the official SatlasPretrain licence notes before redistributing a derived training database.
